# Two-Tower Business Entity Resolution: Kaggle GPU Training & Inference

### Workflow Overview:
1. Attach your preprocessed dataset in Kaggle (`/kaggle/input/...`).
2. Turn on GPU accelerator: **Settings -> Accelerator -> GPU T4 x2 or P100**.
3. Run training with PyTorch Mixed Precision (AMP) for ultra-fast GPU throughput.
4. Trained weights (`best.pt`, `tokenizer.json`, `legal_form_vocab.json`, etc.) are saved to `/kaggle/working/two_tower_model`.
5. An automated zip archive (`two_tower_model.zip`) is created for easy download to your local machine.
6. Verify and run predictions directly in the next cells using the saved weights.


In [16]:
import os
import sys
import time
import json
import re
import shutil
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

# Compute device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[INFO] Using device: {device}")
if device.type == 'cuda':
    print(f"[INFO] GPU Name: {torch.cuda.get_device_name(0)}")

# Helper to automatically locate files in /kaggle/input if dataset name varies
def locate_file(pattern, search_root=Path('/kaggle/input')):
    matches = list(search_root.rglob(pattern)) if search_root.exists() else []
    return matches[0] if matches else None

# Dataset paths: auto-detected from /kaggle/input or set manually
SOURCE1_PATH = locate_file('*sounce1*.tsv') or Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\preprocessed_train_sounce1.tsv')
SOURCE2_PATH = locate_file('*sounce2*.tsv') or Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\preprocessed_train_sounce2.tsv')
SOURCE3_PATH = locate_file('*sounce3*.tsv') or Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\preprocessed_train_sounce3.tsv')
LABELS_PATH  = locate_file('*labeledPairs*.tsv') or Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\labeledPairs.tsv')

OUTPUT_DIR = Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\model_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Training configurations for Kaggle GPU
EPOCHS = 5
BATCH_SIZE = 256  # 256 or 512 for GPU
LEARNING_RATE = 1e-3
MAX_LABEL_ROWS = None  # Use None for full dataset, or e.g. 500000 for quick test

print(f"[INFO] Source 1 Path: {SOURCE1_PATH}")
print(f"[INFO] Source 2 Path: {SOURCE2_PATH}")
print(f"[INFO] Source 3 Path: {SOURCE3_PATH}")
print(f"[INFO] Labels Path  : {LABELS_PATH}")
print(f"[INFO] Output Dir   : {OUTPUT_DIR}")


[INFO] Using device: cpu
[INFO] Source 1 Path: C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\preprocessed_train_sounce1.tsv
[INFO] Source 2 Path: C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\preprocessed_train_sounce2.tsv
[INFO] Source 3 Path: C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\preprocessed_train_sounce3.tsv
[INFO] Labels Path  : C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\pre-processing\labeledPairs.tsv
[INFO] Output Dir   : C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\model_output


In [14]:
# ==============================================================================
# Model Architecture & Utilities (Self-Contained)
# ==============================================================================

TEXT_FIELDS = ("name_norm", "name_ascii", "address_norm", "country_norm")

def split_legal_forms(value: str) -> list[str]:
    value = str(value or "").strip()
    if value.startswith("["):
        try:
            value = ",".join(json.loads(value))
        except (json.JSONDecodeError, TypeError):
            pass
    strip_chars = " '\""
    tokens = re.split(r"[,;|]", value)
    result = []
    for x in tokens:
        cleaned = x.strip(strip_chars)
        if cleaned and cleaned != "missing":
            result.append(cleaned)
    return result



class CharTokenizer:
    PAD, UNK = 0, 1

    def __init__(self, vocab: dict[str, int]):
        self.vocab = vocab

    @classmethod
    def fit(cls, records: pd.DataFrame, max_chars: int = 12000) -> "CharTokenizer":
        counts = Counter()
        for field in TEXT_FIELDS:
            counts.update("".join(records[field].fillna("").astype(str)))
        vocab = {char: index + 2 for index, (char, _) in
                 enumerate(counts.most_common(max_chars))}
        return cls(vocab)

    def encode(self, values, length: int) -> torch.Tensor:
        output = np.zeros((len(values), length), dtype=np.int64)
        for row, value in enumerate(values):
            ids = [self.vocab.get(char, self.UNK) for char in str(value)[:length]]
            output[row, :len(ids)] = ids
        return torch.from_numpy(output)

    def save(self, path):
        Path(path).write_text(json.dumps(self.vocab, ensure_ascii=False), encoding="utf-8")

    @classmethod
    def load(cls, path):
        return cls(json.loads(Path(path).read_text(encoding="utf-8")))


class LegalFormVocab:
    PAD, UNK = 0, 1

    def __init__(self, vocab: dict[str, int]):
        self.vocab = vocab

    @classmethod
    def fit(cls, records: pd.DataFrame) -> "LegalFormVocab":
        values = sorted({form for value in records["legal_form"].fillna("")
                         for form in split_legal_forms(value)})
        return cls({value: index + 2 for index, value in enumerate(values)})

    def encode(self, values) -> torch.Tensor:
        forms = [split_legal_forms(value) for value in values]
        width = max(1, max(map(len, forms), default=0))
        output = np.zeros((len(forms), width), dtype=np.int64)
        for row, values_for_row in enumerate(forms):
            if values_for_row:
                output[row, :len(values_for_row)] = [self.vocab.get(value, self.UNK) for value in values_for_row]
        return torch.from_numpy(output)

    def save(self, path):
        Path(path).write_text(json.dumps(self.vocab, indent=2), encoding="utf-8")

    @classmethod
    def load(cls, path):
        return cls(json.loads(Path(path).read_text(encoding="utf-8")))


class CharCNN(nn.Module):
    def __init__(self, vocab_size: int, char_dim: int = 48, channels: int = 64, output_dim: int = 128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, char_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(char_dim, channels, kernel_size=kernel, padding=kernel // 2)
            for kernel in (3, 5, 7)
        ])
        self.projection = nn.Linear(channels * 3, output_dim)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        present = token_ids.ne(0)
        x = self.embedding(token_ids).transpose(1, 2)
        pooled = []
        for conv in self.convs:
            value = F.relu(conv(x)).masked_fill(~present.unsqueeze(1), -1e4)
            value = value.max(dim=-1).values
            pooled.append(torch.where(present.any(1, keepdim=True), value, torch.zeros_like(value)))
        return self.projection(torch.cat(pooled, dim=1))


class EntityEncoder(nn.Module):
    def __init__(self, char_vocab_size: int, legal_vocab_size: int,
                 char_dim: int = 48, channels: int = 64, field_dim: int = 128,
                 legal_dim: int = 16, output_dim: int = 128, dropout: float = 0.15):
        super().__init__()
        self.name_cnn = CharCNN(char_vocab_size, char_dim, channels, field_dim)
        self.address_cnn = CharCNN(char_vocab_size, char_dim, channels, field_dim)
        self.legal_embedding = nn.Embedding(legal_vocab_size, legal_dim, padding_idx=0)
        self.mlp = nn.Sequential(
            nn.Linear(field_dim * 4 + legal_dim, 256),
            nn.LayerNorm(256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, output_dim),
        )

    def forward(self, batch: dict[str, torch.Tensor]) -> torch.Tensor:
        legal_ids = batch["legal_form"]
        legal_vectors = self.legal_embedding(legal_ids)
        legal_mask = legal_ids.ne(0).unsqueeze(-1)
        legal = (legal_vectors * legal_mask).sum(1) / legal_mask.sum(1).clamp_min(1)
        features = torch.cat([
            self.name_cnn(batch["name_norm"]),
            self.name_cnn(batch["name_ascii"]),
            self.address_cnn(batch["address_norm"]),
            self.address_cnn(batch["country_norm"]),
            legal,
        ], dim=1)
        return F.normalize(self.mlp(features), p=2, dim=1)


def multi_positive_infonce_loss(query_embeddings, candidate_embeddings,
                                query_ids, candidate_owner_ids, temperature: float = 0.07):
    logits = (query_embeddings @ candidate_embeddings.T) / temperature
    positive = query_ids[:, None].eq(candidate_owner_ids[None, :])
    if not positive.any(dim=1).all():
        raise ValueError("Every query needs at least one positive candidate in its batch.")
    positive_score = torch.logsumexp(logits.masked_fill(~positive, -torch.inf), dim=1)
    all_score = torch.logsumexp(logits, dim=1)
    return (all_score - positive_score).mean()


def cosine_triplet_loss(query_embedding, positive_embedding, negative_embedding, margin: float = 0.2):
    positive_similarity = (query_embedding * positive_embedding).sum(1)
    negative_similarity = (query_embedding * negative_embedding).sum(1)
    return F.relu(margin + negative_similarity - positive_similarity).mean()


def records_to_batch(records: pd.DataFrame, tokenizer: CharTokenizer,
                     legal_vocab: LegalFormVocab, device, name_length: int = 112,
                     address_length: int = 224, country_length: int = 32):
    return {
        "name_norm": tokenizer.encode(records["name_norm"], name_length).to(device),
        "name_ascii": tokenizer.encode(records["name_ascii"], name_length).to(device),
        "address_norm": tokenizer.encode(records["address_norm"], address_length).to(device),
        "country_norm": tokenizer.encode(records["country_norm"], country_length).to(device),
        "legal_form": legal_vocab.encode(records["legal_form"]).to(device),
    }


class PositivePairDataset(Dataset):
    def __init__(self, positives: pd.DataFrame, records: pd.DataFrame, negatives: pd.DataFrame):
        self.left = records.loc[positives.source1_entity_id].reset_index(drop=True)
        self.positive = records.loc[positives.target_entity_id].reset_index(drop=True)
        negative_lookup = defaultdict(list)
        for row in negatives.itertuples(index=False):
            negative_lookup[row.source1_entity_id].append(row.target_entity_id)
        self.negative_ids = [values[0] if values else None for values in
                             (negative_lookup[q] for q in positives.source1_entity_id)]
        self.records = records

    def __len__(self):
        return len(self.left)

    def __getitem__(self, index):
        negative = self.records.loc[self.negative_ids[index]] if self.negative_ids[index] else None
        return self.left.iloc[index], self.positive.iloc[index], negative


def collate_pairs(items):
    left, positive, negative = zip(*items)
    negative_indices = [index for index, row in enumerate(negative) if row is not None]
    return (pd.DataFrame(left), pd.DataFrame(positive),
            pd.DataFrame([row for row in negative if row is not None]), negative_indices)


def split_by_source1(positive_labels, validation_fraction: float = 0.2, seed: int = 42):
    ids = np.array(sorted(positive_labels.source1_entity_id.unique()))
    rng = np.random.default_rng(seed)
    rng.shuffle(ids)
    validation_ids = set(ids[:max(1, int(len(ids) * validation_fraction))])
    return (positive_labels[~positive_labels.source1_entity_id.isin(validation_ids)].copy(),
            positive_labels[positive_labels.source1_entity_id.isin(validation_ids)].copy())


@torch.inference_mode()
def generate_embeddings(model, tokenizer, legal_vocab, records: pd.DataFrame, device, batch_size: int = 512):
    output = []
    model.eval()
    for start in range(0, len(records), batch_size):
        batch = records_to_batch(records.iloc[start:start + batch_size], tokenizer, legal_vocab, device)
        output.append(model(batch).cpu())
    return torch.cat(output).numpy() if output else np.empty((0, 128), dtype=np.float32)

print("[INFO] Model architecture and helper functions loaded.")


[INFO] Model architecture and helper functions loaded.


In [10]:
# ==============================================================================
# Fast Data Loading (Optimized Columns & Memory)
# ==============================================================================
COLS = ["entity_id", "name_norm", "name_ascii", "address_norm", "country_norm", "legal_form"]

print("[INFO] 1. Loading Source 1...")
t0 = time.time()
df_s1 = pd.read_csv(SOURCE1_PATH, sep="\t", usecols=COLS, dtype=str).fillna("")
print(f"       Done in {time.time()-t0:.1f}s | Rows: {len(df_s1):,}")

print("[INFO] 2. Loading Source 2...")
t0 = time.time()
df_s2 = pd.read_csv(SOURCE2_PATH, sep="\t", usecols=COLS, dtype=str).fillna("")
print(f"       Done in {time.time()-t0:.1f}s | Rows: {len(df_s2):,}")

print("[INFO] 3. Loading Source 3...")
t0 = time.time()
df_s3 = pd.read_csv(SOURCE3_PATH, sep="\t", usecols=COLS, dtype=str).fillna("")
print(f"       Done in {time.time()-t0:.1f}s | Rows: {len(df_s3):,}")

print("[INFO] 4. Loading Labeled Pairs...")
t0 = time.time()
df_labels = pd.read_csv(LABELS_PATH, sep="\t", nrows=MAX_LABEL_ROWS, dtype=str, keep_default_na=False)
df_labels["label"] = df_labels["label"].astype(float).astype(int)
print(f"       Done in {time.time()-t0:.1f}s | Rows: {len(df_labels):,}")

print("[INFO] Combining records in memory...")
t0 = time.time()
records = pd.concat([df_s1, df_s2, df_s3], ignore_index=True)
del df_s1, df_s2, df_s3  # Free memory
records = records.drop_duplicates("entity_id").set_index("entity_id", drop=False)

labels = df_labels[
    df_labels["source1_entity_id"].isin(records.index) & 
    df_labels["target_entity_id"].isin(records.index)
].copy()
del df_labels

print(f"[INFO] Dataset ready in {time.time()-t0:.1f}s")
print(f"       Total Unique Records : {len(records):,}")
print(f"       Total Labeled Pairs  : {len(labels):,}")
print(f"       Positive Pairs (1)   : {int((labels.label == 1).sum()):,}")
print(f"       Negative Pairs (0)   : {int((labels.label == 0).sum()):,}")


[INFO] 1. Loading Source 1...
       Done in 5.7s | Rows: 2,206,821
[INFO] 2. Loading Source 2...
       Done in 20.5s | Rows: 5,034,616
[INFO] 3. Loading Source 3...
       Done in 19.5s | Rows: 5,285,603
[INFO] 4. Loading Labeled Pairs...
       Done in 41.6s | Rows: 44,526,560
[INFO] Combining records in memory...
[INFO] Dataset ready in 137.6s
       Total Unique Records : 12,527,040
       Total Labeled Pairs  : 44,526,560
       Positive Pairs (1)   : 7,638,364
       Negative Pairs (0)   : 36,888,196


In [12]:
# # ==============================================================================
# # Model Training (GPU Mixed Precision + Deferred Disk Writes)
# # ==============================================================================
# positives = labels[labels.label == 1].copy()
# negatives = labels[labels.label == 0].copy()

# if positives.empty:
#     raise ValueError("At least one positive pair is required.")

# train_pos, validation_pos = split_by_source1(positives)
# train_qids = set(train_pos.source1_entity_id)
# train_neg = negatives[negatives.source1_entity_id.isin(train_qids)]
# validation_neg = negatives[~negatives.source1_entity_id.isin(train_qids)]

# train_record_ids = list(set(train_pos.source1_entity_id) | set(train_pos.target_entity_id))
# train_records = records.loc[train_record_ids]

# print("[INFO] Fitting Tokenizer and Legal Form Vocabularies...")
# tokenizer = CharTokenizer.fit(train_records)
# legal_vocab = LegalFormVocab.fit(train_records)

# model = EntityEncoder(len(tokenizer.vocab) + 2, len(legal_vocab.vocab) + 2).to(device)
# optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
# scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

# loaders = {
#     "train": DataLoader(PositivePairDataset(train_pos, records, train_neg), batch_size=BATCH_SIZE,
#                         shuffle=True, collate_fn=collate_pairs, num_workers=2, pin_memory=(device.type == "cuda")),
#     "validation": DataLoader(PositivePairDataset(validation_pos, records, validation_neg), batch_size=BATCH_SIZE,
#                              shuffle=False, collate_fn=collate_pairs, num_workers=2, pin_memory=(device.type == "cuda")),
# }

# print(f"\n[INFO] Starting Training on Device: {device}")
# print(f"[INFO] Train Batches: {len(loaders['train']):,} | Validation Batches: {len(loaders['validation']):,}")
# print(f"[INFO] Total Epochs: {EPOCHS} | Batch Size: {BATCH_SIZE} | Learning Rate: {LEARNING_RATE}\n")

# history = []
# best_loss = float("inf")
# best_state_dict = None
# best_epoch = 1

# for epoch in range(1, EPOCHS + 1):
#     values = {}
#     for stage, loader in loaders.items():
#         training = stage == "train"
#         model.train(training)
#         losses = []
#         desc = f"Epoch [{epoch:02d}/{EPOCHS:02d}] {stage.capitalize()}"
#         pbar = tqdm(loader, desc=desc, leave=False, dynamic_ncols=True)
#         for left, positive, negative, negative_indices in pbar:
#             if training:
#                 optimizer.zero_grad(set_to_none=True)
#             with torch.set_grad_enabled(training), torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
#                 q = model(records_to_batch(left, tokenizer, legal_vocab, device))
#                 p = model(records_to_batch(positive, tokenizer, legal_vocab, device))
#                 owner = torch.as_tensor(left.entity_id.astype("category").cat.codes.to_numpy(copy=True), device=device)
#                 candidate_owner = owner
#                 loss = multi_positive_infonce_loss(q, p, owner, candidate_owner)
#                 if not negative.empty:
#                     n = model(records_to_batch(negative, tokenizer, legal_vocab, device))
#                     selected = torch.as_tensor(negative_indices, device=device)
#                     loss = loss + 0.25 * cosine_triplet_loss(q[selected], p[selected], n)
#             if training:
#                 scaler.scale(loss).backward()
#                 scaler.unscale_(optimizer)
#                 torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
#                 scaler.step(optimizer)
#                 scaler.update()
#             batch_loss = float(loss.detach().cpu())
#             losses.append(batch_loss)
#             pbar.set_postfix({"batch_loss": f"{batch_loss:.4f}", "avg_loss": f"{np.mean(losses):.4f}"})
#         values[f"{stage}_loss"] = float(np.mean(losses))
#     values["epoch"] = epoch
#     history.append(values)
    
#     if values["validation_loss"] < best_loss:
#         best_loss = values["validation_loss"]
#         best_epoch = epoch
#         best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
#         best_marker = "[BEST SO FAR]"
#     else:
#         best_marker = ""
        
#     print(f"Epoch [{epoch:02d}/{EPOCHS:02d}]  |  "
#           f"Train Loss: {values['train_loss']:.4f}  |  "
#           f"Val Loss: {values['validation_loss']:.4f}  {best_marker}")

# # ==============================================================================
# # Save Weights & Artifacts (Only once after complete training)
# # ==============================================================================
# print(f"\n[INFO] Complete training finished. Saving final weights to: {OUTPUT_DIR}")
# if best_state_dict is not None:
#     torch.save({"model": best_state_dict, "epoch": best_epoch, "best_loss": best_loss}, OUTPUT_DIR / "best.pt")
# torch.save({"model": model.state_dict(), "epoch": EPOCHS, "history": history}, OUTPUT_DIR / "last.pt")
# tokenizer.save(OUTPUT_DIR / "tokenizer.json")
# legal_vocab.save(OUTPUT_DIR / "legal_form_vocab.json")
# (OUTPUT_DIR / "config.json").write_text(json.dumps({"output_dim": 128}, indent=2), encoding="utf-8")
# (OUTPUT_DIR / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")

# # Create zip archive for 1-click download from Kaggle outputs
# zip_path = Path("/kaggle/working/two_tower_model")
# shutil.make_archive(str(zip_path), 'zip', OUTPUT_DIR)
# print(f"[INFO] Created zip archive for download: {zip_path}.zip")
# print("[INFO] Model weights and vocabularies saved successfully.")


In [17]:
# ==============================================================================
# Load Saved Weights from Disk (Verification)
# ==============================================================================
print(f"[INFO] Loading saved weights from: {OUTPUT_DIR.resolve()}")

# 1. Load Vocabularies
eval_tokenizer = CharTokenizer.load(OUTPUT_DIR / "tokenizer.json")
eval_legal_vocab = LegalFormVocab.load(OUTPUT_DIR / "legal_form_vocab.json")

# 2. Re-initialize Model & Load State Dict
eval_model = EntityEncoder(
    char_vocab_size=len(eval_tokenizer.vocab) + 2,
    legal_vocab_size=len(eval_legal_vocab.vocab) + 2
).to(device)

checkpoint = torch.load(OUTPUT_DIR / "best.pt", map_location=device)
eval_model.load_state_dict(checkpoint["model"])
eval_model.eval()

print(f"[INFO] Successfully loaded model checkpoint from Best Epoch: {checkpoint.get('epoch', 'N/A')}")
print(f"[INFO] Best Validation Loss: {checkpoint.get('best_loss', 'N/A'):.4f}")


[INFO] Loading saved weights from: C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\model_output
[INFO] Successfully loaded model checkpoint from Best Epoch: 4
[INFO] Best Validation Loss: 0.0022


C:\Users\ps302\AppData\Local\Temp\ipykernel_36920\3380833075.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(OUTPUT_DIR / "best.pt", map_locatio

In [14]:
!pip install faiss-gpu

In [15]:
# import faiss
# import gc

# print("[INFO] Preparing candidate records for FAISS streaming...")

# target_ids = labels["target_entity_id"].unique()
# candidates_df = records.loc[records.index.isin(target_ids)].copy()
# total_candidates = len(candidates_df)
# print(f"[INFO] Total candidate records to index: {total_candidates:,}")

# # Initialize FAISS Index (Exact Inner Product = Cosine Similarity)
# embedding_dim = 128
# faiss_index = faiss.IndexFlatIP(embedding_dim)

# # Save candidate entity IDs to disk to save RAM
# candidate_ids = candidates_df["entity_id"].to_numpy()
# np.save(OUTPUT_DIR / "candidate_ids.npy", candidate_ids)

# # Stream in chunks of 50,000 to keep RAM usage under 1.5 GB
# CHUNK_SIZE = 50000
# INFERENCE_BATCH_SIZE = 1024

# print(f"\n[INFO] Streaming {total_candidates:,} candidates into FAISS in chunks of {CHUNK_SIZE:,}...")
# t_start = time.time()

# eval_model.eval()
# with torch.inference_mode():
#     for start_idx in range(0, total_candidates, CHUNK_SIZE):
#         end_idx = min(start_idx + CHUNK_SIZE, total_candidates)
#         chunk_df = candidates_df.iloc[start_idx:end_idx]
        
#         # Batch inference on GPU
#         chunk_embeddings = []
#         for b_start in range(0, len(chunk_df), INFERENCE_BATCH_SIZE):
#             batch_slice = chunk_df.iloc[b_start:b_start + INFERENCE_BATCH_SIZE]
#             batch_inputs = records_to_batch(batch_slice, eval_tokenizer, eval_legal_vocab, device)
            
#             with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
#                 batch_emb = eval_model(batch_inputs)
            
#             chunk_embeddings.append(batch_emb.cpu().numpy().astype(np.float32))
        
#         # Add directly to FAISS C++ memory
#         chunk_matrix = np.concatenate(chunk_embeddings, axis=0)
#         faiss_index.add(chunk_matrix)
        
#         # Free memory immediately
#         del chunk_embeddings, chunk_matrix
#         gc.collect()
#         if device.type == "cuda":
#             torch.cuda.empty_cache()
            
#         pct = (end_idx / total_candidates) * 100
#         print(f"  Indexed [{end_idx:,}/{total_candidates:,}] ({pct:.1f}%) | FAISS Total: {faiss_index.ntotal:,}")

# t_elapsed = time.time() - t_start
# print(f"\n[SUCCESS] FAISS indexing complete in {t_elapsed:.1f}s!")

# # Save FAISS Index to disk
# faiss_file = OUTPUT_DIR / "candidate_index.faiss"
# faiss.write_index(faiss_index, str(faiss_file))
# print(f"[INFO] FAISS index saved to disk: {faiss_file}")


In [18]:
import numpy as np
import faiss
from pathlib import Path

# Paths define karein
CANDIDATE_IDS_PATH = Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\model_output\candidate_ids.npy')
FAISS_INDEX_PATH = Path(r'C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\data\model_output\candidate_index.faiss')

# 1. Load Candidate IDs
print("Loading Candidate IDs...")
candidate_ids = np.load(CANDIDATE_IDS_PATH, allow_pickle=True)

# 2. Reconstruct candidates_df as a Pandas DataFrame
# Ye line IDs ke basis par global 'records' DataFrame se original DataFrame bana degi
print("Reconstructing candidates_df DataFrame...")
candidates_df = records.loc[candidate_ids].copy()

# 3. Load FAISS Index
print("Loading FAISS Index...")
faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))

print(f"Loaded successfully! Total indexed candidates: {len(candidate_ids):,}")

Loading Candidate IDs...
Reconstructing candidates_df DataFrame...
Loading FAISS Index...
Loaded successfully! Total indexed candidates: 9,895,200


In [ ]:
def predict_matches_faiss(query_id: str, top_k: int = 5):
    """Retrieves Top-K candidate matches in < 1ms using FAISS index without brute-force matrix multiply."""
    if query_id not in records.index:
        print(f"[ERROR] Query ID '{query_id}' not found in records.")
        return None

    # 1. Encode single query record on GPU
    query_row = records.loc[[query_id]]
    eval_model.eval()
    with torch.inference_mode():
        batch_inputs = records_to_batch(query_row, eval_tokenizer, eval_legal_vocab, device)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            query_emb = eval_model(batch_inputs).cpu().numpy().astype(np.float32)

    # 2. FAISS Index Search (Ultra-fast C++ Inner Product search)
    t0 = time.time()
    scores, indices = faiss_index.search(query_emb, top_k)
    search_time_ms = (time.time() - t0) * 1000

    scores = scores[0]
    indices = indices[0]

    # Ground truth verification
    actual_positives = set(
        labels[(labels["source1_entity_id"] == query_id) & (labels["label"] == 1)]["target_entity_id"]
    )

    q = query_row.iloc[0]
    print("\n" + "=" * 70)
    print(f"QUERY ID   : {query_id}")
    print(f"Name       : {q.get('name_norm', '')}")
    print(f"Address    : {q.get('address_norm', '')}")
    print(f"Country    : {q.get('country_norm', '')}")
    print(f"Search Time: {search_time_ms:.2f} ms")
    print("=" * 70)
    print(f"FAISS TOP {top_k} PREDICTED MATCHES:")
    print("=" * 70)

    for rank, (cand_idx, score) in enumerate(zip(indices, scores), start=1):
        cand_id = candidate_ids[cand_idx]
        cand_row = candidates_df.iloc[cand_idx]
        match_flag = "[GROUND TRUTH MATCH]" if cand_id in actual_positives else ""

        print(f"Rank {rank} | Score: {score:.4f} {match_flag}")
        print(f"  ID        : {cand_id}")
        print(f"  Name      : {cand_row.get('name_norm', '')}")
        print(f"  Address   : {cand_row.get('address_norm', '')}")
        print(f"  Country   : {cand_row.get('country_norm', '')}")
        print(f"  Legal Form: {cand_row.get('legal_form', '')}")
        print("-" * 70)


# Test with sample positive query
sample_query = labels[labels["label"] == 1].iloc[6]["source1_entity_id"]
predict_matches_faiss(sample_query, top_k=10)



QUERY ID   : S1-773889195
Name       : prime money
Address    : 17560 ellis road tahlequah ok
Country    : us
Search Time: 629.74 ms
FAISS TOP 5 PREDICTED MATCHES:
Rank 1 | Score: 0.9937 [GROUND TRUTH MATCH]
  ID        : S2-374107005
  Name      : prime money
  Address   : tahlequah ok 0017560 ellis rd
  Country   : us
  Legal Form: missing
----------------------------------------------------------------------
Rank 2 | Score: 0.9923 [GROUND TRUTH MATCH]
  ID        : S3-161452820
  Name      : prime money
  Address   : 17560 ellis rd tahlequah oklahoma
  Country   : us
  Legal Form: incorporated
----------------------------------------------------------------------
Rank 3 | Score: 0.9916 [GROUND TRUTH MATCH]
  ID        : S3-189140669
  Name      : prime money
  Address   : 17560 ellis rd tahlequah oklahoma
  Country   : us
  Legal Form: missing
----------------------------------------------------------------------
Rank 4 | Score: 0.9760 [GROUND TRUTH MATCH]
  ID        : S3-62223287

In [28]:
import time
import torch

def predict_matches_faiss(query_id: str, top_k: int = 10):
    """Retrieves Top-K candidate matches in < 1ms using FAISS index without brute-force matrix multiply."""
    if query_id not in records.index:
        print(f"[ERROR] Query ID '{query_id}' not found in records.")
        return None

    # 1. Encode single query record on GPU/CPU
    query_row = records.loc[[query_id]]
    eval_model.eval()
    
    # torch.no_grad() is safer on Intel GPUs than inference_mode()
    with torch.no_grad():
        batch_inputs = records_to_batch(query_row, eval_tokenizer, eval_legal_vocab, device)
        # Assuming you're not strictly using CUDA here, autocast is removed for broad compatibility
        query_emb = eval_model(batch_inputs).cpu().numpy().astype(np.float32)

    # 2. FAISS Index Search (Ultra-fast C++ Inner Product search)
    t0 = time.time()
    scores, indices = faiss_index.search(query_emb, top_k)
    search_time_ms = (time.time() - t0) * 1000

    scores = scores[0]
    indices = indices[0]

    # Ground truth verification
    actual_positives = set(
        labels[(labels["source1_entity_id"] == query_id) & (labels["label"] == 1)]["target_entity_id"]
    )

    q = query_row.iloc[0]
    print("\n" + "=" * 70)
    print(f"QUERY ID   : {query_id}")
    print(f"Name       : {q.get('name_norm', '')}")
    print(f"Address    : {q.get('address_norm', '')}")
    print(f"Country    : {q.get('country_norm', '')}")
    print(f"Search Time: {search_time_ms:.2f} ms")
    print("=" * 70)
    print(f"FAISS TOP {top_k} PREDICTED MATCHES:")
    print("=" * 70)

    for rank, (cand_idx, score) in enumerate(zip(indices, scores), start=1):
        cand_id = candidate_ids[cand_idx]
        
        # Changed candidates_df.iloc to records.loc to avoid undefined variable error
        cand_row = records.loc[cand_id] if cand_id in records.index else {}
        match_flag = "[GROUND TRUTH MATCH]" if cand_id in actual_positives else ""

        print(f"Rank {rank} | Score: {score:.4f} {match_flag}")
        print(f"  ID        : {cand_id}")
        print(f"  Name      : {cand_row.get('name_norm', '')}")
        print(f"  Address   : {cand_row.get('address_norm', '')}")
        print(f"  Country   : {cand_row.get('country_norm', '')}")
        print(f"  Legal Form: {cand_row.get('legal_form', '')}")
        print("-" * 70)


# ==============================================================================
# Extract 5 Queries DIRECTLY from Source 1 and get 10 Recommendations
# ==============================================================================
print("Getting 5 queries directly from Source 1 data...")

# Finding the first 5 records whose ID starts with 'S1-'
source1_queries = [idx for idx in records.index if str(idx).startswith("S1-")][:5]

for i, query_id in enumerate(source1_queries, start=1):
    print(f"\n---> Running Test {i}/5 <---")
    # Yahan top_k = 10 pass kar rahe hain
    predict_matches_faiss(query_id, top_k=10)


Getting 5 queries directly from Source 1 data...

---> Running Test 1/5 <---

QUERY ID   : S1-925783039
Name       : orelees barbershop
Address    : 1795 westchester drive high point nc
Country    : us
Search Time: 564.65 ms
FAISS TOP 10 PREDICTED MATCHES:
Rank 1 | Score: 0.9999 [GROUND TRUTH MATCH]
  ID        : S2-517291332
  Name      : orelees bárbershop
  Address   : 1795 westchester drive high point nc
  Country   : us
  Legal Form: missing
----------------------------------------------------------------------
Rank 2 | Score: 0.9974 [GROUND TRUTH MATCH]
  ID        : S2-157377754
  Name      : orelees barbershop
  Address   : 1795 westchester drive nc high point
  Country   : us
  Legal Form: missing
----------------------------------------------------------------------
Rank 3 | Score: 0.9839 [GROUND TRUTH MATCH]
  ID        : S3-997698194
  Name      : orelees barbershop
  Address   : 1795 westchester dr north carolina high point
  Country   : us
  Legal Form: missing
----------

In [20]:
import pandas as pd
from pathlib import Path

# Ground Truth data ka path jo aapne diya
GROUND_TRUTH_PATH = r"C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\dataset\train\train_ground_truth.tsv"

print(f"Loading Ground Truth data from: {GROUND_TRUTH_PATH}")
# Tab-separated data load kar rahe hain
gt_df = pd.read_csv(GROUND_TRUTH_PATH, sep="\t")

def fetch_ground_truth_matches(query_id: str):
    """Bina kisi model ke seedha 'labels' dataframe se actual ground truth matches nikalta hai."""
    
    if query_id not in records.index:
        print(f"[ERROR] Query ID '{query_id}' not found in records memory.")
        return
        
    query_row = records.loc[query_id]
    
    # Jaisa FAISS code me tha, wahi logic ground truth nikalne ke liye
    actual_positives = set(
        labels[(labels["source1_entity_id"] == query_id) & (labels["label"] == 1)]["target_entity_id"]
    )
    
    print("\n" + "=" * 70)
    print(f"QUERY ID   : {query_id}")
    print(f"Name       : {query_row.get('name_norm', '')}")
    print(f"Address    : {query_row.get('address_norm', '')}")
    print(f"Country    : {query_row.get('country_norm', '')}")
    print("=" * 70)
    print(f"ACTUAL GROUND TRUTH MATCHES FOUND: {len(actual_positives)}")
    print("=" * 70)
    
    if not actual_positives:
        print("No matches found for this query in the Ground Truth labels.")
        return
        
    for i, cand_id in enumerate(actual_positives, start=1):
        if cand_id in records.index:
            cand_row = records.loc[cand_id]
            print(f"Match {i} | [ACTUAL GROUND TRUTH]")
            print(f"  ID        : {cand_id}")
            print(f"  Name      : {cand_row.get('name_norm', '')}")
            print(f"  Address   : {cand_row.get('address_norm', '')}")
            print(f"  Country   : {cand_row.get('country_norm', '')}")
            print(f"  Legal Form: {cand_row.get('legal_form', '')}")
        else:
            print(f"Match {i} | [ACTUAL GROUND TRUTH]")
            print(f"  ID        : {cand_id} (Details not available in records memory)")
        print("-" * 70)


# ==============================================================================
# Extract 5 Queries DIRECTLY from Source 1 and Print Actual Ground Truths
# ==============================================================================
print("Getting 5 queries directly from Source 1 data...")

# Finding the first 5 records whose ID starts with 'S1-'
source1_queries = [idx for idx in records.index if str(idx).startswith("S1-")][:5]

for i, query_id in enumerate(source1_queries, start=1):
    print(f"\n---> Running Ground Truth Lookup Test {i}/5 <---")
    fetch_ground_truth_matches(query_id)


# ==============================================================================
# Extract 5 Queries DIRECTLY from Source 1 and Print Actual Ground Truths
# ==============================================================================
print("Getting 5 queries directly from Source 1 data...")

# Finding the first 5 records whose ID starts with 'S1-'
source1_queries = [idx for idx in records.index if str(idx).startswith("S1-")][:5]

for i, query_id in enumerate(source1_queries, start=1):
    print(f"\n---> Running Ground Truth Lookup Test {i}/5 <---")
    fetch_ground_truth_matches(query_id)


Loading Ground Truth data from: C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\dataset\train\train_ground_truth.tsv
Getting 5 queries directly from Source 1 data...

---> Running Ground Truth Lookup Test 1/5 <---

QUERY ID   : S1-925783039
Name       : orelees barbershop
Address    : 1795 westchester drive high point nc
Country    : us
ACTUAL GROUND TRUTH MATCHES FOUND: 4
Match 1 | [ACTUAL GROUND TRUTH]
  ID        : S2-517291332
  Name      : orelees bárbershop
  Address   : 1795 westchester drive high point nc
  Country   : us
  Legal Form: missing
----------------------------------------------------------------------
Match 2 | [ACTUAL GROUND TRUTH]
  ID        : S3-997698194
  Name      : orelees barbershop
  Address   : 1795 westchester dr north carolina high point
  Country   : us
  Legal Form: missing
----------------------------------------------------------------------
Match 3 | [ACTUAL GROUND TRUTH]
  ID        : S2-157377754
  Name      : orelees barbershop

In [21]:
import time
import torch

def search_model_by_id(query_id: str, top_k: int = 10):
    """Deep Learning Model (EntityEncoder) aur FAISS ka use karke top matches nikalta hai."""
    
    if query_id not in records.index:
        print(f"[ERROR] Query ID '{query_id}' not found in records dataframe.")
        return

    print(f"🔄 Processing Query ID: {query_id} on {device}...")
    
    # 1. Query ka data nikalein
    query_row = records.loc[[query_id]]
    eval_model.eval()
    
    # 2. Model se Neural Embeddings banayein
    t0 = time.time()
    with torch.no_grad():
        batch_inputs = records_to_batch(query_row, eval_tokenizer, eval_legal_vocab, device)
        query_emb = eval_model(batch_inputs).cpu().numpy().astype(np.float32)
    t_emb = (time.time() - t0) * 1000

    # 3. FAISS Index me Nearest Neighbors search karein
    t0 = time.time()
    scores, indices = faiss_index.search(query_emb, top_k)
    t_faiss = (time.time() - t0) * 1000

    scores = scores[0]
    indices = indices[0]

    # Ground truth verification ke liye labels check karein
    actual_positives = set(
        labels[(labels["source1_entity_id"] == query_id) & (labels["label"] == 1)]["target_entity_id"]
    ) if 'labels' in globals() else set()

    # Query ki details print karein
    q = query_row.iloc[0]
    print("\n" + "=" * 70)
    print(f"🔎 YOUR QUERY DETAILS")
    print("=" * 70)
    print(f"ID         : {query_id}")
    print(f"Name       : {q.get('name_norm', '')}")
    print(f"Address    : {q.get('address_norm', '')}")
    print(f"Country    : {q.get('country_norm', '')}")
    print("-" * 70)
    print(f"⏱️ Model Encoding : {t_emb:.2f} ms")
    print(f"⏱️ FAISS Search   : {t_faiss:.2f} ms")
    
    print("\n" + "=" * 70)
    print(f"🎯 TOP {top_k} MODEL RECOMMENDATIONS")
    print("=" * 70)

    # Recommendations print karein
    for rank, (cand_idx, score) in enumerate(zip(indices, scores), start=1):
        cand_id = candidate_ids[cand_idx]
        cand_row = records.loc[cand_id] if cand_id in records.index else {}
        
        # Agar ye candidate actual match hai toh highlight karein
        match_flag = "✅ [GROUND TRUTH MATCH]" if cand_id in actual_positives else "❌"

        print(f"Rank {rank} | Score: {score:.4f} {match_flag}")
        print(f"  ID        : {cand_id}")
        print(f"  Name      : {cand_row.get('name_norm', '')}")
        print(f"  Address   : {cand_row.get('address_norm', '')}")
        print(f"  Country   : {cand_row.get('country_norm', '')}")
        print(f"  Legal Form: {cand_row.get('legal_form', '')}")
        print("-" * 70)


# ==============================================================================
# 🚀 Yahan apna Query ID daalein
# ==============================================================================
CUSTOM_QUERY_ID = "S1-19826348"  # <--- Ise change karke koi bhi S1 ID daal sakte hain

search_model_by_id(CUSTOM_QUERY_ID, top_k=10)


🔄 Processing Query ID: S1-19826348 on cpu...

🔎 YOUR QUERY DETAILS
ID         : S1-19826348
Name       : uptown grill
Address    : 340 aragon road tucson az
Country    : us
----------------------------------------------------------------------
⏱️ Model Encoding : 220.69 ms
⏱️ FAISS Search   : 10382.23 ms

🎯 TOP 10 MODEL RECOMMENDATIONS
Rank 1 | Score: 0.9988 ✅ [GROUND TRUTH MATCH]
  ID        : S2-814759322
  Name      : uptown grill
  Address   : az 340 aragon road tucson
  Country   : us
  Legal Form: llc
----------------------------------------------------------------------
Rank 2 | Score: 0.9946 ✅ [GROUND TRUTH MATCH]
  ID        : S2-124586126
  Name      : uptown grill grill
  Address   : az tucson 340 aragon road
  Country   : us
  Legal Form: llc
----------------------------------------------------------------------
Rank 3 | Score: 0.9861 ✅ [GROUND TRUTH MATCH]
  ID        : S2-88675544
  Name      : the uptown grill
  Address   : 340 aragon road tucson az
  Country   : us
  Le

In [13]:
import pandas as pd
from pathlib import Path

# Ground Truth data ka path
GROUND_TRUTH_PATH = r"C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\dataset\train\train_ground_truth.tsv"

# Bar-bar load hone se bachane ke liye sirf ek baar load karenge
if 'gt_df' not in globals():
    print(f"Loading Ground Truth data from: {GROUND_TRUTH_PATH} ...")
    gt_df = pd.read_csv(GROUND_TRUTH_PATH, sep="\t")

def search_ground_truth_by_id(query_id: str):
    """Bina Model ke seedha ground truth file se candidate details print karta hai."""
    
    if query_id not in records.index:
        print(f"[ERROR] Query ID '{query_id}' not found in records dataframe.")
        return
        
    print(f"🔄 Searching ACTUAL matches for: {query_id} ...")
    
    query_row = records.loc[query_id]
    
    # Columns ke naam safe-check
    s1_col = 'source1_entity_id' if 'source1_entity_id' in gt_df.columns else gt_df.columns[0]
    t_col = 'target_entity_id' if 'target_entity_id' in gt_df.columns else gt_df.columns[1]
    
    # Data filter karna
    matches = gt_df[gt_df[s1_col] == query_id]
    
    if 'label' in gt_df.columns:
        matches = matches[matches['label'] == 1]
        
    candidate_ids = matches[t_col].tolist()
    
    # Query ki details print karna
    print("\n" + "=" * 70)
    print(f"🔎 YOUR QUERY DETAILS")
    print("=" * 70)
    print(f"ID         : {query_id}")
    print(f"Name       : {query_row.get('name_norm', '')}")
    print(f"Address    : {query_row.get('address_norm', '')}")
    print(f"Country    : {query_row.get('country_norm', '')}")
    
    print("\n" + "=" * 70)
    print(f"✅ ACTUAL GROUND TRUTH MATCHES: {len(candidate_ids)}")
    print("=" * 70)
    
    if not candidate_ids:
        print("❌ No matches found for this query in the Ground Truth file.")
        return
        
    # Matches ki details print karna
    for i, cand_id in enumerate(candidate_ids, start=1):
        if cand_id in records.index:
            cand_row = records.loc[cand_id]
            print(f"Match {i}")
            print(f"  ID        : {cand_id}")
            print(f"  Name      : {cand_row.get('name_norm', '')}")
            print(f"  Address   : {cand_row.get('address_norm', '')}")
            print(f"  Country   : {cand_row.get('country_norm', '')}")
            print(f"  Legal Form: {cand_row.get('legal_form', '')}")
        else:
            print(f"Match {i} | ID: {cand_id} (Details not available in records memory)")
        print("-" * 70)

# ==============================================================================
# 🚀 Yahan apna Query ID daalein
# ==============================================================================
CUSTOM_QUERY_ID = "S1-19826348"  # <--- Ise change karke koi bhi ID daal sakte hain

search_ground_truth_by_id(CUSTOM_QUERY_ID)


🔄 Searching ACTUAL matches for: S1-19826348 ...

🔎 YOUR QUERY DETAILS
ID         : S1-19826348
Name       : uptown grill
Address    : 340 aragon road tucson az
Country    : us

✅ ACTUAL GROUND TRUTH MATCHES: 1
Match 1 | ID: S2-733319472,S2-814759322,S2-124586126,S2-88675544 (Details not available in records memory)
----------------------------------------------------------------------


In [23]:
import time
import random
import torch
import pandas as pd

def compare_model_vs_ground_truth(query_id: str, top_k: int = 10):
    if query_id not in records.index:
        return

    # 1. Get Query Details
    query_row = records.loc[query_id]
    q_name = query_row.get('name_norm', '')
    
    # 2. Get Actual Ground Truth Matches
    actual_positives = set(
        labels[(labels["source1_entity_id"] == query_id) & (labels["label"] == 1)]["target_entity_id"]
    )
    
    # 3. Get Model Predictions via FAISS
    eval_model.eval()
    with torch.no_grad():
        batch_inputs = records_to_batch(records.loc[[query_id]], eval_tokenizer, eval_legal_vocab, device)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            query_emb = eval_model(batch_inputs).cpu().numpy().astype(np.float32)

    scores, indices = faiss_index.search(query_emb, top_k)
    scores = scores[0]
    indices = indices[0]
    
    predicted_candidates = [candidate_ids[idx] for idx in indices]
    
    # ================= TABLE PRINTING =================
    print("\n" + "=" * 90)
    print(f"🔍 QUERY : {query_id} | Name: {q_name}")
    # NEW: Print total actual recommendations in Ground Truth
    print(f"✅ Total Ground Truth Matches Available: {len(actual_positives)}")
    print("=" * 90)
    print(f"{'TYPE':<12} | {'RANK':<5} | {'CANDIDATE ID':<15} | {'SCORE':<7} | {'MATCH?':<8} | {'CANDIDATE NAME'}")
    print("-" * 90)
    
    # A. Print Model Predictions
    for rank, (cand_id, score) in enumerate(zip(predicted_candidates, scores), start=1):
        cand_name = records.loc[cand_id].get('name_norm', '') if cand_id in records.index else "Unknown"
        
        is_match = cand_id in actual_positives
        match_str = "✅ YES" if is_match else "❌ NO"
        
        print(f"{'Model':<12} | {rank:<5} | {cand_id:<15} | {score:.4f}  | {match_str:<8} | {cand_name[:40]}")
        
    # B. Print Ground Truths MISSED by Model (If any)
    missed_gts = actual_positives - set(predicted_candidates)
    if missed_gts:
        print("-" * 90)
        for cand_id in missed_gts:
            cand_name = records.loc[cand_id].get('name_norm', '') if cand_id in records.index else "Unknown"
            print(f"{'GT (Missed)':<12} | {'-':<5} | {cand_id:<15} | {'N/A':<7} | {'✅ MISSED':<8} | {cand_name[:40]}")
    
    print("=" * 90)


# ==============================================================================
# Run Comparison on 10 Random Source 1 Queries
# ==============================================================================
print("Selecting 10 random Source 1 queries for side-by-side comparison...\n")

# Extract all Source 1 IDs
all_source1_ids = [idx for idx in records.index if str(idx).startswith("S1-")]

# Select 10 random queries (Seed fix for reproducibility)
random.seed(42) 
random_queries = random.sample(all_source1_ids, 10)

for i, q_id in enumerate(random_queries, start=1):
    print(f"\n---> Table {i}/10 <---")
    compare_model_vs_ground_truth(q_id, top_k=10)


Selecting 10 random Source 1 queries for side-by-side comparison...


---> Table 1/10 <---

🔍 QUERY : S1-602781481 | Name: agra care
✅ Total Ground Truth Matches Available: 5
TYPE         | RANK  | CANDIDATE ID    | SCORE   | MATCH?   | CANDIDATE NAME
------------------------------------------------------------------------------------------
Model        | 1     | S3-605262587    | 0.9892  | ✅ YES    | agra care
Model        | 2     | S2-259582547    | 0.9862  | ✅ YES    | agra cáre
Model        | 3     | S2-324794612    | 0.9384  | ✅ YES    | agracare com
Model        | 4     | S2-478553828    | 0.8563  | ❌ NO     | agra area limited
Model        | 5     | S2-499720943    | 0.8478  | ❌ NO     | agra servicse limited
Model        | 6     | S3-559045239    | 0.8421  | ❌ NO     | agra bangalore
Model        | 7     | S2-794662589    | 0.8420  | ❌ NO     | agra bangalore
Model        | 8     | S3-545524327    | 0.8400  | ❌ NO     | agra bangalore
Model        | 9     | S2-259309883    | 0.

In [24]:
import time
import torch
import pandas as pd

def compare_model_vs_ground_truth(query_id: str, top_k: int = 10):
    if query_id not in records.index:
        return

    # 1. Get Query Details
    query_row = records.loc[query_id]
    q_name = query_row.get('name_norm', '')
    
    # 2. Get Actual Ground Truth Matches
    actual_positives = set(
        labels[(labels["source1_entity_id"] == query_id) & (labels["label"] == 1)]["target_entity_id"]
    )
    
    # 3. Get Model Predictions via FAISS
    eval_model.eval()
    with torch.no_grad():
        batch_inputs = records_to_batch(records.loc[[query_id]], eval_tokenizer, eval_legal_vocab, device)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            query_emb = eval_model(batch_inputs).cpu().numpy().astype(np.float32)

    scores, indices = faiss_index.search(query_emb, top_k)
    scores = scores[0]
    indices = indices[0]
    
    predicted_candidates = [candidate_ids[idx] for idx in indices]
    
    # ================= TABLE PRINTING =================
    print("\n" + "=" * 90)
    print(f"🔍 QUERY : {query_id} | Name: {q_name}")
    print(f"✅ Total Ground Truth Matches Available: {len(actual_positives)}")
    print("=" * 90)
    print(f"{'TYPE':<12} | {'RANK':<5} | {'CANDIDATE ID':<15} | {'SCORE':<7} | {'MATCH?':<8} | {'CANDIDATE NAME'}")
    print("-" * 90)
    
    # A. Print Model Predictions
    for rank, (cand_id, score) in enumerate(zip(predicted_candidates, scores), start=1):
        cand_name = records.loc[cand_id].get('name_norm', '') if cand_id in records.index else "Unknown"
        
        is_match = cand_id in actual_positives
        match_str = "✅ YES" if is_match else "❌ NO"
        
        print(f"{'Model':<12} | {rank:<5} | {cand_id:<15} | {score:.4f}  | {match_str:<8} | {cand_name[:40]}")
        
    # B. Print Ground Truths MISSED by Model (If any)
    missed_gts = actual_positives - set(predicted_candidates)
    if missed_gts:
        print("-" * 90)
        for cand_id in missed_gts:
            cand_name = records.loc[cand_id].get('name_norm', '') if cand_id in records.index else "Unknown"
            print(f"{'GT (Missed)':<12} | {'-':<5} | {cand_id:<15} | {'N/A':<7} | {'✅ MISSED':<8} | {cand_name[:40]}")
    
    print("=" * 90)


# ==============================================================================
# Run Comparison on a Specific RANGE of Source 1 Queries
# ==============================================================================
# Extract all Source 1 IDs (Ek baar list nikal li)
all_source1_ids = [idx for idx in records.index if str(idx).startswith("S1-")]

# Yahan par apni range set karein (e.g., pehli 10 queries ke liye 0 se 10)
START_INDEX = 0
END_INDEX = 10

print(f"Selecting queries from index {START_INDEX} to {END_INDEX} for 100% reproducibility...\n")
range_queries = all_source1_ids[START_INDEX:END_INDEX]

for i, q_id in enumerate(range_queries, start=START_INDEX + 1):
    print(f"\n---> Table (Index: {i}) <---")
    compare_model_vs_ground_truth(q_id, top_k=10)


Selecting queries from index 0 to 10 for 100% reproducibility...


---> Table (Index: 1) <---

🔍 QUERY : S1-925783039 | Name: orelees barbershop
✅ Total Ground Truth Matches Available: 4
TYPE         | RANK  | CANDIDATE ID    | SCORE   | MATCH?   | CANDIDATE NAME
------------------------------------------------------------------------------------------
Model        | 1     | S2-517291332    | 0.9999  | ✅ YES    | orelees bárbershop
Model        | 2     | S2-157377754    | 0.9974  | ✅ YES    | orelees barbershop
Model        | 3     | S3-997698194    | 0.9839  | ✅ YES    | orelees barbershop
Model        | 4     | S2-335180728    | 0.8383  | ❌ NO     | orelees barbershop downtown
Model        | 5     | S3-698172821    | 0.7988  | ✅ YES    | orelees services
Model        | 6     | S2-49558376     | 0.7551  | ❌ NO     | poole and safe pokhrel associates harbor
Model        | 7     | S3-50207325     | 0.7036  | ❌ NO     | golden barbershop
Model        | 8     | S2-712302391    | 0.7020  |

In [25]:
import pandas as pd

def search_entity_details(entity_id: str):
    """ID ke hisaab se pata lagata hai ki kis Source me dekhna hai aur details lata hai"""
    
    print("\n" + "=" * 70)
    print(f"🔍 SEARCHING DETAILS FOR : {entity_id}")
    print("=" * 70)

    # 1. Prefix se Source ka pata lagana
    if entity_id.startswith("S1-"):
        print("📂 Origin: SOURCE 1 (sounce1.tsv)")
        target_df_name = 'df_s1'
    elif entity_id.startswith("S2-"):
        print("📂 Origin: SOURCE 2 (sounce2.tsv)")
        target_df_name = 'df_s2'
    elif entity_id.startswith("S3-"):
        print("📂 Origin: SOURCE 3 (sounce3.tsv)")
        target_df_name = 'df_s3'
    else:
        print("📂 Origin: UNKNOWN SOURCE")
        target_df_name = None

    row = None
    
    # 2. Agar separate dataframe (df_s1, df_s2) abhi RAM me zinda hai, toh wahan se search karo
    if target_df_name and target_df_name in globals():
        df = globals()[target_df_name]
        # Dataframe ko filter kar ke record nikalna
        match = df[df['entity_id'] == entity_id]
        if not match.empty:
            row = match.iloc[0]
            print("✅ Found in individual source dataframe!")
            
    # 3. Agar separate DF delete ho gaye hain, toh combined 'records' dataframe se search karo
    if row is None and 'records' in globals():
        if entity_id in records.index:
            row = records.loc[entity_id]
            print("✅ Found in combined 'records' memory!")
            
    # 4. Agar kahin nahi mila
    if row is None:
        print(f"❌ [ERROR] '{entity_id}' could not be found anywhere in memory.")
        print("=" * 70)
        return

    # 5. Result ko ekdum neat table format me print karna
    print("-" * 70)
    print(f"  Entity ID  : {row.get('entity_id', entity_id)}")
    print(f"  Name (Norm): {row.get('name_norm', '')}")
    print(f"  Name ASCII : {row.get('name_ascii', '')}")
    print(f"  Address    : {row.get('address_norm', '')}")
    print(f"  Country    : {row.get('country_norm', '')}")
    print(f"  Legal Form : {row.get('legal_form', '')}")
    print("=" * 70)


# ==============================================================================
# 🚀 Yahan apna Entity ID daalein (S1, S2, ya S3 kisi ka bhi)
# ==============================================================================
search_entity_details("S1-851869949")  # Source 2 search
search_entity_details("S3-366891958")   # Source 1 search
search_entity_details("S3-807524060")  # Source 3 search



🔍 SEARCHING DETAILS FOR : S1-851869949
📂 Origin: SOURCE 1 (sounce1.tsv)
✅ Found in combined 'records' memory!
----------------------------------------------------------------------
  Entity ID  : S1-851869949
  Name (Norm): custom wealth services
  Name ASCII : custom wealth services
  Address    : oh columbus 5559 orville avenue
  Country    : us
  Legal Form : llc

🔍 SEARCHING DETAILS FOR : S3-366891958
📂 Origin: SOURCE 3 (sounce3.tsv)
✅ Found in combined 'records' memory!
----------------------------------------------------------------------
  Entity ID  : S3-366891958
  Name (Norm): custom wealth services partners
  Name ASCII : custom wealth services partners
  Address    : 5564 orville avenue columus cdp ohio
  Country    : us
  Legal Form : missing

🔍 SEARCHING DETAILS FOR : S3-807524060
📂 Origin: SOURCE 3 (sounce3.tsv)
✅ Found in combined 'records' memory!
----------------------------------------------------------------------
  Entity ID  : S3-807524060
  Name (Norm): custom w

In [26]:
import time
import torch
import pandas as pd

def compare_single_id_vs_ground_truth(query_id: str, top_k: int = 10):
    if query_id not in records.index:
        print(f"[ERROR] Query ID '{query_id}' not found in records dataframe.")
        return

    # 1. Query Details Fetch Karein
    query_row = records.loc[query_id]
    q_name = query_row.get('name_norm', '')
    
    # 2. Actual Ground Truth Matches Nikalein
    actual_positives = set(
        labels[(labels["source1_entity_id"] == query_id) & (labels["label"] == 1)]["target_entity_id"]
    )
    
    # 3. Model se Predictions (FAISS) Nikalein
    eval_model.eval()
    with torch.no_grad():
        batch_inputs = records_to_batch(records.loc[[query_id]], eval_tokenizer, eval_legal_vocab, device)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            query_emb = eval_model(batch_inputs).cpu().numpy().astype(np.float32)

    scores, indices = faiss_index.search(query_emb, top_k)
    scores = scores[0]
    indices = indices[0]
    
    predicted_candidates = [candidate_ids[idx] for idx in indices]
    
    # ================= TABLE PRINTING =================
    print("\n" + "=" * 90)
    print(f"🔍 CUSTOM QUERY : {query_id} | Name: {q_name}")
    print(f"✅ Total Ground Truth Matches Available: {len(actual_positives)}")
    print("=" * 90)
    print(f"{'TYPE':<12} | {'RANK':<5} | {'CANDIDATE ID':<15} | {'SCORE':<7} | {'MATCH?':<8} | {'CANDIDATE NAME'}")
    print("-" * 90)
    
    # A. Print Model Predictions
    for rank, (cand_id, score) in enumerate(zip(predicted_candidates, scores), start=1):
        cand_name = records.loc[cand_id].get('name_norm', '') if cand_id in records.index else "Unknown"
        
        is_match = cand_id in actual_positives
        match_str = "✅ YES" if is_match else "❌ NO"
        
        print(f"{'Model':<12} | {rank:<5} | {cand_id:<15} | {score:.4f}  | {match_str:<8} | {cand_name[:40]}")
        
    # B. Print Ground Truths MISSED by Model (Agar koi chhut gaya ho)
    missed_gts = actual_positives - set(predicted_candidates)
    if missed_gts:
        print("-" * 90)
        for cand_id in missed_gts:
            cand_name = records.loc[cand_id].get('name_norm', '') if cand_id in records.index else "Unknown"
            print(f"{'GT (Missed)':<12} | {'-':<5} | {cand_id:<15} | {'N/A':<7} | {'✅ MISSED':<8} | {cand_name[:40]}")
    
    print("=" * 90)


# ==============================================================================
# 🚀 Yahan apna CUSTOM Query ID daalein
# ==============================================================================
CUSTOM_QUERY_ID = "S1-851869949"  # <--- Aap isko change kar sakte hain

compare_single_id_vs_ground_truth(CUSTOM_QUERY_ID, top_k=10)



🔍 CUSTOM QUERY : S1-851869949 | Name: custom wealth services
✅ Total Ground Truth Matches Available: 5
TYPE         | RANK  | CANDIDATE ID    | SCORE   | MATCH?   | CANDIDATE NAME
------------------------------------------------------------------------------------------
Model        | 1     | S2-915265864    | 0.9873  | ✅ YES    | custom wealth partners
Model        | 2     | S2-961342067    | 0.9873  | ✅ YES    | custom wealth
Model        | 3     | S2-801813753    | 0.9791  | ✅ YES    | custom wealth services
Model        | 4     | S3-366891958    | 0.9740  | ❌ NO     | custom wealth services partners
Model        | 5     | S3-949828938    | 0.8810  | ✅ YES    | custom services services
Model        | 6     | S3-266262731    | 0.7929  | ❌ NO     | south coastal smith
Model        | 7     | S2-881135158    | 0.7922  | ❌ NO     | south coastal smith
Model        | 8     | S2-395433291    | 0.7810  | ❌ NO     | cust0m hh
Model        | 9     | S2-106287141    | 0.7693  | ❌ NO     | cus

In [1]:
# import time
# import pandas as pd
# import numpy as np
# import torch
# from pathlib import Path
# from tqdm.auto import tqdm

# # Output Path
# output_csv = r"/kaggle/working/ann_top_10.csv"
# top_k = 10
# batch_size = 1024

# # Fallback just in case DEVICE is not defined
# try:
#     DEVICE
# except NameError:
#     DEVICE = torch.device('cpu')

# print("1. Preparing positive labels for fast lookup...")
# # Create a fast O(1) lookup set of positive pairs (source1_id, target_id)
# positive_pairs = set(
#     zip(
#         labels[labels["label"] == 1]["source1_entity_id"],
#         labels[labels["label"] == 1]["target_entity_id"]
#     )
# )

# print("2. Extracting all Source 1 IDs from records...")
# # Fast extraction of Source 1 IDs from the records dataframe
# source1_ids = [idx for idx in records.index if str(idx).startswith("S1-")]
# total_queries = len(source1_ids)
# print(f"Total Source 1 queries to process: {total_queries:,}")

# # Prepare output CSV file with headers
# output_path = Path(output_csv)
# output_path.parent.mkdir(parents=True, exist_ok=True)
# with open(output_path, "w", encoding="utf-8") as f:
#     f.write("query_id,candidate_id,label\n")

# eval_model.eval()
# print(f"\nStarting FAISS batch extraction & saving to {output_csv}...")

# # Process in Batches
# with torch.no_grad():
#     for start_idx in tqdm(range(0, total_queries, batch_size), desc="Extracting ANN Top-10"):
#         end_idx = min(start_idx + batch_size, total_queries)
#         batch_query_ids = source1_ids[start_idx:end_idx]
        
#         # 1. Generate Embeddings for batch
#         batch_df = records.loc[batch_query_ids]
#         batch_inputs = records_to_batch(batch_df, eval_tokenizer, eval_legal_vocab, DEVICE)
#         query_embs = eval_model(batch_inputs).cpu().numpy().astype(np.float32)
        
#         # 2. FAISS Search
#         scores, indices = faiss_index.search(query_embs, top_k)
        
#         # 3. Format results and attach Ground Truth Labels
#         batch_results = []
#         for i, q_id in enumerate(batch_query_ids):
#             for rank in range(top_k):
#                 cand_idx = indices[i, rank]
#                 cand_id = candidate_ids[cand_idx]
                
#                 # Check if this pair is a known positive match in our labels
#                 label = 1 if (q_id, cand_id) in positive_pairs else 0
                    
#                 batch_results.append(f"{q_id},{cand_id},{label}\n")
        
#         # 4. Append to CSV
#         with open(output_path, "a", encoding="utf-8") as f:
#             f.writelines(batch_results)

# print("✅ Batch processing complete! Data saved successfully for LightGBM Reranker.")


In [4]:
import pandas as pd
import numpy as np
import time
from rapidfuzz import fuzz, distance

# ==============================================================================
# Helper Functions
# ==============================================================================
def get_ngrams(text, n):
    text = str(text)
    if len(text) < n: return {text}
    return set([text[i:i+n] for i in range(len(text)-n+1)])

def get_tokens(text):
    return str(text).split() if pd.notna(text) and str(text).strip() else []

def get_token_metrics(t1_list, t2_list):
    t1, t2 = set(t1_list), set(t2_list)
    if not t1 and not t2: return 0.0, 0.0, 0.0
    if not t1 or not t2: return 0.0, 0.0, 0.0
    
    intersection = len(t1.intersection(t2))
    union = len(t1.union(t2))
    
    overlap = float(intersection)
    jaccard = intersection / union if union > 0 else 0.0
    containment = intersection / min(len(t1), len(t2)) if min(len(t1), len(t2)) > 0 else 0.0
    
    return overlap, jaccard, containment

def ngram_similarity(text1, text2, n):
    ng1, ng2 = get_ngrams(text1, n), get_ngrams(text2, n)
    if not ng1 and not ng2: return 0.0
    if not ng1 or not ng2: return 0.0
    return len(ng1.intersection(ng2)) / max(1, len(ng1.union(ng2)))

# ==============================================================================
# Main Feature Builder
# ==============================================================================
def build_features_for_pair(query_id, candidate_id, records_df):
    row1 = records_df.loc[query_id].fillna("")
    row2 = records_df.loc[candidate_id].fillna("")
    
    name1, name2 = str(row1['name_norm']), str(row2['name_norm'])
    ascii1, ascii2 = str(row1['name_ascii']), str(row2['name_ascii'])
    legal1, legal2 = str(row1['legal_form']), str(row2['legal_form'])
    country1, country2 = str(row1['country_norm']), str(row2['country_norm'])
    
    name_tokens1, name_tokens2 = get_tokens(name1), get_tokens(name2)
    n_overlap, n_jaccard, n_contain = get_token_metrics(name_tokens1, name_tokens2)
    
    # Legal form presence checks
    l1_present = legal1 not in ["", "missing"]
    l2_present = legal2 not in ["", "missing"]
    
    features = {
        'query_id': query_id,
        'candidate_id': candidate_id,
        
        # ---------------- NAME ----------------
        'exact_name_norm': int(name1 == name2 and name1 != ""),
        'exact_name_ascii': int(ascii1 == ascii2 and ascii1 != ""),
        
        'fuzzy_ratio': fuzz.ratio(name1, name2) / 100.0,
        'fuzzy_partial_ratio': fuzz.partial_ratio(name1, name2) / 100.0,
        'fuzzy_token_sort_ratio': fuzz.token_sort_ratio(name1, name2) / 100.0,
        'fuzzy_token_set_ratio': fuzz.token_set_ratio(name1, name2) / 100.0,
        'fuzzy_WRatio': fuzz.WRatio(name1, name2) / 100.0,
        'jaro_winkler': distance.JaroWinkler.similarity(name1, name2),
        
        'token_overlap': n_overlap,
        'token_jaccard': n_jaccard,
        'token_containment': n_contain,
        
        'char_2gram_similarity': ngram_similarity(name1, name2, 2),
        'char_3gram_similarity': ngram_similarity(name1, name2, 3),
        'char_4gram_similarity': ngram_similarity(name1, name2, 4),
        'char_5gram_similarity': ngram_similarity(name1, name2, 5),
        
        'name_length_difference': abs(len(name1) - len(name2)),
        'name_length_ratio': min(len(name1), len(name2)) / max(1, max(len(name1), len(name2))),
        'name_token_count_diff': abs(len(name_tokens1) - len(name_tokens2)),
        
        'first_token_match': int((name_tokens1[0] == name_tokens2[0]) if name_tokens1 and name_tokens2 else 0),
        'last_token_match': int((name_tokens1[-1] == name_tokens2[-1]) if name_tokens1 and name_tokens2 else 0),
        
        # ---------------- LEGAL FORM ----------------
        'legal_form_missing': int(not l1_present and not l2_present),
        'both_legal_form_present': int(l1_present and l2_present),
        'one_legal_form_present': int(l1_present != l2_present),
        'same_legal_form': int(legal1 == legal2 and l1_present),
        'legal_form_match': fuzz.ratio(legal1, legal2) / 100.0 if (l1_present or l2_present) else 0.0,
        
        # ---------------- COUNTRY ----------------
        'country_equal': int(country1 == country2 and country1 != "")
    }
    
    return features


# ==============================================================================
# TIMED EXTRACTION TEST (5 ROWS)
# ==============================================================================
print("Loading top 5 rows from ann_top_10.csv...")
sample_pairs = pd.read_csv(r"C:\Users\ps302\OneDrive\Desktop\Amazon\business-entity-resolution\src\Reranker\preprocessed_data\ann_top_10.csv", nrows=5)

feature_rows = []
total_time = 0.0

print("\nStarting Feature Extraction...")
for idx, row in sample_pairs.iterrows():
    q_id = row['query_id']
    c_id = row['candidate_id']
    
    t0 = time.perf_counter()
    
    f_dict = build_features_for_pair(q_id, c_id, records)
    
    # --- ANN / MODEL ---
    f_dict['ann_rank'] = row.get('ann_rank', 0)
    f_dict['ann_similarity'] = row.get('ann_similarity', 0.0)
    f_dict['label'] = row['label']
    
    exec_time = (time.perf_counter() - t0) * 1000
    total_time += exec_time
    
    print(f"Row {idx+1} | {q_id} <-> {c_id} | Time Taken: {exec_time:.2f} ms")
    feature_rows.append(f_dict)

print("-" * 50)
print(f"Total Time for 5 Rows  : {total_time:.2f} ms")
print(f"Average Time per Query : {total_time / len(sample_pairs):.2f} ms")
print("-" * 50)

features_df = pd.DataFrame(feature_rows)
pd.set_option('display.max_columns', None) 
display(features_df)


Loading top 5 rows from ann_top_10.csv...

Starting Feature Extraction...
Row 1 | S1-925783039 <-> S2-517291332 | Time Taken: 4809.53 ms
Row 2 | S1-925783039 <-> S2-157377754 | Time Taken: 0.51 ms
Row 3 | S1-925783039 <-> S3-997698194 | Time Taken: 0.26 ms
Row 4 | S1-925783039 <-> S2-335180728 | Time Taken: 0.63 ms
Row 5 | S1-925783039 <-> S3-698172821 | Time Taken: 0.38 ms
--------------------------------------------------
Total Time for 5 Rows  : 4811.30 ms
Average Time per Query : 962.26 ms
--------------------------------------------------


,query_id,candidate_id,exact_name_norm,exact_name_ascii,fuzzy_ratio,fuzzy_partial_ratio,fuzzy_token_sort_ratio,fuzzy_token_set_ratio,fuzzy_WRatio,jaro_winkler,token_overlap,token_jaccard,token_containment,char_2gram_similarity,char_3gram_similarity,char_4gram_similarity,char_5gram_similarity,name_length_difference,name_length_ratio,name_token_count_diff,first_token_match,last_token_match,legal_form_missing,both_legal_form_present,one_legal_form_present,same_legal_form,legal_form_match,country_equal,ann_rank,ann_similarity,label
0,S1-925783039,S2-517291332,0,1,0.944444,0.944444,0.944444,0.944444,0.944444,0.977778,1.0,0.333333,0.5,0.789474,0.684211,0.578947,0.473684,0,1.000000,0,1,0,1,0,0,0,0.0,1,1,0.9999,1
1,S1-925783039,S2-157377754,1,1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.0,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,0,1.000000,0,1,1,1,0,0,0,0.0,1,2,0.9974,1
2,S1-925783039,S3-997698194,1,1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.0,1.000000,1.0,1.000000,1.000000,1.000000,1.000000,0,1.000000,0,1,1,1,0,0,0,0.0,1,3,0.9839,1
3,S1-925783039,S2-335180728,0,0,0.800000,1.000000,0.800000,1.000000,0.900000,0.933333,2.0,0.666667,1.0,0.708333,0.666667,0.625000,0.608696,9,0.666667,1,1,0,1,0,0,0,0.0,1,4,0.8383,0
4,S1-925783039,S3-698172821,0,0,0.647059,0.709677,0.411765,0.647059,0.647059,0.841540,1.0,0.333333,0.5,0.347826,0.250000,0.217391,0.181818,2,0.888889,0,1,0,1,0,0,0,0.0,1,5,0.7987,1


In [7]:
len(sample_pairs)

5

In [ ]:
import os
import zipfile
import pandas as pd
import numpy as np
import time
from tqdm.auto import tqdm
from joblib import Parallel, delayed
from rapidfuzz import fuzz, distance
# ==============================================================================
# Helper Functions (Keep these as they are)
# ==============================================================================
def get_ngrams(text, n):
    text = str(text)
    if len(text) < n: return {text}
    return set([text[i:i+n] for i in range(len(text)-n+1)])
def get_tokens(text):
    return str(text).split() if pd.notna(text) and str(text).strip() else []
def get_token_metrics(t1_list, t2_list):
    t1, t2 = set(t1_list), set(t2_list)
    if not t1 and not t2: return 0.0, 0.0, 0.0
    if not t1 or not t2: return 0.0, 0.0, 0.0
    
    intersection = len(t1.intersection(t2))
    union = len(t1.union(t2))
    
    overlap = float(intersection)
    jaccard = intersection / union if union > 0 else 0.0
    containment = intersection / min(len(t1), len(t2)) if min(len(t1), len(t2)) > 0 else 0.0
    
    return overlap, jaccard, containment
def ngram_similarity(text1, text2, n):
    ng1, ng2 = get_ngrams(text1, n), get_ngrams(text2, n)
    if not ng1 and not ng2: return 0.0
    if not ng1 or not ng2: return 0.0
    return len(ng1.intersection(ng2)) / max(1, len(ng1.union(ng2)))
# ==============================================================================
# Single Row Processor (Called by multiple threads)
# ==============================================================================
def process_single_row(query_id, candidate_id, ann_rank, ann_sim, label):
    # Lookup data from global 'records' dataframe
    row1 = records.loc[query_id].fillna("")
    row2 = records.loc[candidate_id].fillna("")
    
    name1, name2 = str(row1['name_norm']), str(row2['name_norm'])
    ascii1, ascii2 = str(row1['name_ascii']), str(row2['name_ascii'])
    legal1, legal2 = str(row1['legal_form']), str(row2['legal_form'])
    country1, country2 = str(row1['country_norm']), str(row2['country_norm'])
    
    name_tokens1, name_tokens2 = get_tokens(name1), get_tokens(name2)
    n_overlap, n_jaccard, n_contain = get_token_metrics(name_tokens1, name_tokens2)
    
    l1_present = legal1 not in ["", "missing"]
    l2_present = legal2 not in ["", "missing"]
    
    features = {
        'query_id': query_id,
        'candidate_id': candidate_id,
        
        # --- NAME ---
        'exact_name_norm': int(name1 == name2 and name1 != ""),
        'exact_name_ascii': int(ascii1 == ascii2 and ascii1 != ""),
        'fuzzy_ratio': fuzz.ratio(name1, name2) / 100.0,
        'fuzzy_partial_ratio': fuzz.partial_ratio(name1, name2) / 100.0,
        'fuzzy_token_sort_ratio': fuzz.token_sort_ratio(name1, name2) / 100.0,
        'fuzzy_token_set_ratio': fuzz.token_set_ratio(name1, name2) / 100.0,
        'fuzzy_WRatio': fuzz.WRatio(name1, name2) / 100.0,
        'jaro_winkler': distance.JaroWinkler.similarity(name1, name2),
        'token_overlap': n_overlap,
        'token_jaccard': n_jaccard,
        'token_containment': n_contain,
        'char_2gram_similarity': ngram_similarity(name1, name2, 2),
        'char_3gram_similarity': ngram_similarity(name1, name2, 3),
        'char_4gram_similarity': ngram_similarity(name1, name2, 4),
        'char_5gram_similarity': ngram_similarity(name1, name2, 5),
        'name_length_difference': abs(len(name1) - len(name2)),
        'name_length_ratio': min(len(name1), len(name2)) / max(1, max(len(name1), len(name2))),
        'name_token_count_diff': abs(len(name_tokens1) - len(name_tokens2)),
        'first_token_match': int((name_tokens1[0] == name_tokens2[0]) if name_tokens1 and name_tokens2 else 0),
        'last_token_match': int((name_tokens1[-1] == name_tokens2[-1]) if name_tokens1 and name_tokens2 else 0),
        
        # --- LEGAL FORM ---
        'legal_form_missing': int(not l1_present and not l2_present),
        'both_legal_form_present': int(l1_present and l2_present),
        'one_legal_form_present': int(l1_present != l2_present),
        'same_legal_form': int(legal1 == legal2 and l1_present),
        'legal_form_match': fuzz.ratio(legal1, legal2) / 100.0 if (l1_present or l2_present) else 0.0,
        
        # --- COUNTRY ---
        'country_equal': int(country1 == country2 and country1 != ""),
        
        # --- ANN / MODEL ---
        'ann_rank': ann_rank,
        'ann_similarity': ann_sim,
        'label': label
    }
    
    return features
# ==============================================================================
# BATCH PROCESSING WITH MULTITHREADING
# ==============================================================================
input_csv = "/kaggle/working/ann_top_10.csv"
output_csv = "/kaggle/working/training_features_full.csv"
output_zip = "/kaggle/working/training_features_full.zip"
# Chunk size (kitni rows ek baar me RAM me aayengi)
CHUNK_SIZE = 250_000 
TOTAL_ROWS = 22_068_211
total_chunks = (TOTAL_ROWS // CHUNK_SIZE) + 1
num_threads = os.cpu_count()  # Automatically use all CPU cores
print(f"[INFO] Processing {TOTAL_ROWS:,} rows in {total_chunks} batches using {num_threads} CPU Threads...")
t_start_total = time.time()
# File ko truncate kar dete hain agar pehle se bani ho
if os.path.exists(output_csv):
    os.remove(output_csv)
header_written = False
# Read CSV in chunks
chunk_iterator = pd.read_csv(input_csv, chunksize=CHUNK_SIZE)
for chunk in tqdm(chunk_iterator, total=total_chunks, desc="Building Features"):
    # Parallel thread processing inside the chunk
    # RapidFuzz is C++ based, so it releases GIL. Multithreading makes it extremely fast.
    chunk_results = Parallel(n_jobs=num_threads, prefer="threads")(
        delayed(process_single_row)(
            row.query_id, row.candidate_id, row.ann_rank, row.ann_similarity, row.label
        ) for row in chunk.itertuples(index=False)
    )
    
    # List of Dicts ko DataFrame me convert kiya
    features_df = pd.DataFrame(chunk_results)
    
    # CSV ke end me chipka diya (Append Mode)
    features_df.to_csv(output_csv, mode='a', header=not header_written, index=False)
    header_written = True
t_end_total = time.time()
print(f"\n✅ All Features built successfully in {(t_end_total - t_start_total)/60:.1f} minutes!")
# ==============================================================================
# ZIP COMPRESSION
# ==============================================================================
print(f"\n[INFO] Compressing {output_csv} to ZIP file...")
with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(output_csv, arcname="training_features_full.csv")
print(f"🎉 Compression Complete! You can now download: {output_zip}")

In [ ]:
import time
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import faiss
from pathlib import Path
from tqdm.auto import tqdm

output_csv = "/kaggle/working/ann_top_10.csv"
top_k = 10
# 2 GPUs hone ki wajah se hum batch size double kar sakte hain!
batch_size = 8192 

print("1. Preparing positive labels for fast lookup...")
positive_pairs = set(
    zip(
        labels[labels["label"] == 1]["source1_entity_id"],
        labels[labels["label"] == 1]["target_entity_id"]
    )
)

print("2. Extracting all Source 1 IDs from records...")
source1_ids = [idx for idx in records.index if str(idx).startswith("S1-")]
total_queries = len(source1_ids)
print(f"Total Source 1 queries to process: {total_queries:,}")

# =========================================================
# MULTI-GPU ACCELERATION: FAISS & PYTORCH
# =========================================================
num_gpus = torch.cuda.device_count()

# 1. Multi-GPU for FAISS
if num_gpus > 0:
    print(f"3. Detected {num_gpus} GPUs! Replicating FAISS Index to ALL GPUs...")
    # Ye function FAISS index ko automatically dono GPUs me copy kar dega
    faiss_search_index = faiss.index_cpu_to_all_gpus(faiss_index) 
    print("   [SUCCESS] FAISS Index is now running in Multi-GPU mode!")
else:
    print("3. CUDA not found, using CPU FAISS index.")
    faiss_search_index = faiss_index

# 2. Multi-GPU for PyTorch
if num_gpus > 1 and not isinstance(eval_model, nn.DataParallel):
    print("4. Wrapping PyTorch Model in DataParallel to split load across GPUs...")
    # Ye model ki processing aadhi-aadhi dono GPUs ko baant dega
    eval_model = nn.DataParallel(eval_model)
    model_device = torch.device("cuda")
else:
    model_device = next(eval_model.parameters()).device
# =========================================================

output_path = Path(output_csv)
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, "w", encoding="utf-8") as f:
    f.write("query_id,candidate_id,label\n")

eval_model.eval()
print(f"\nStarting DUAL-GPU FAISS extraction & saving to {output_csv}...")

with torch.no_grad():
    for start_idx in tqdm(range(0, total_queries, batch_size), desc="Extracting ANN Top-10"):
        end_idx = min(start_idx + batch_size, total_queries)
        batch_query_ids = source1_ids[start_idx:end_idx]
        
        batch_df = records.loc[batch_query_ids]
        
        # PyTorch Embedding Generation (Split across 2 GPUs automatically)
        batch_inputs = records_to_batch(batch_df, eval_tokenizer, eval_legal_vocab, model_device)
        with torch.autocast(device_type="cuda" if num_gpus>0 else "cpu", enabled=num_gpus > 0):
            query_embs = eval_model(batch_inputs).cpu().numpy().astype(np.float32)
        
        # FAISS Index Search (Split queries across 2 GPUs automatically)
        scores, indices = faiss_search_index.search(query_embs, top_k)
        
        batch_results = []
        for i, q_id in enumerate(batch_query_ids):
            for rank in range(top_k):
                cand_idx = indices[i, rank]
                cand_id = candidate_ids[cand_idx]
                
                label = 1 if (q_id, cand_id) in positive_pairs else 0
                batch_results.append(f"{q_id},{cand_id},{label}\n")
        
        with open(output_path, "a", encoding="utf-8") as f:
            f.writelines(batch_results)

print("✅ Dual-GPU Batch processing complete! Data saved successfully.")
